lest start by building SCM generator form tabPFN itslef

In [57]:
import torch
import torch.nn as nn
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

def generate_random_dag(num_nodes, edge_prob=0.3):
    adj_matrix = np.random.rand(num_nodes, num_nodes) < edge_prob
    adj_matrix = np.triu(adj_matrix, k=1)  # Ensure acyclic
    return nx.from_numpy_array(adj_matrix ,create_using=nx.DiGraph)

In [58]:
# --- 1. THE MECHANISM ---
class CausalMechanism(nn.Module):
    def __init__(self, n_parents):
        super().__init__()
        if n_parents == 0:
            self.model = None
        else:
            self.model = nn.Sequential(
                nn.Linear(n_parents, 16),
                nn.ReLU(),
                nn.Linear(16, 1)
            )
            for param in self.parameters():
                param.requires_grad = False

    def forward(self, x, noise):
        if self.model is None:
            return noise
        return self.model(x).view(-1) + noise

In [59]:
# --- 2. MULTIVERSE GENERATOR ---
def generate_full_multiverse(n_samples, n_features, do_val=5.0):
    # Pure Torch generation to avoid Numpy errors
    adj = (torch.rand(n_features, n_features) < 0.3).triu(diagonal=1)
    mechanisms = [CausalMechanism(int(adj[:, i].sum())) for i in range(n_features)]
    exogenous_noise = torch.randn(n_samples, n_features)
    
    multiverse = torch.zeros(n_features + 1, n_samples, n_features)

    # Universe 0 (Obs)
    for i in range(n_features):
        parents = torch.where(adj[:, i])[0]
        noise = exogenous_noise[:, i]
        p_data = multiverse[0, :, parents] if len(parents) > 0 else None
        multiverse[0, :, i] = mechanisms[i](p_data, noise)

    # Universes 1-D (Int)
    for u_idx in range(1, n_features + 1):
        target_node = u_idx - 1
        for i in range(n_features):
            parents = torch.where(adj[:, i])[0]
            noise = exogenous_noise[:, i]
            if i == target_node:
                multiverse[u_idx, :, i] = do_val
            else:
                p_data = multiverse[u_idx, :, parents] if len(parents) > 0 else None
                multiverse[u_idx, :, i] = mechanisms[i](p_data, noise)
                
    return multiverse, adj

In [60]:
# --- FINAL EXPERIMENT SUITE ---

# 1. Update the Embedding with the 'Causal Hint'
class ParallelUniverseEmbedding(nn.Module):
    def __init__(self, n_features, embed_dim=128):
        super().__init__()
        self.value_encoder = nn.Linear(1, embed_dim)
        self.feature_embed = nn.Embedding(n_features, embed_dim)
        self.universe_embed = nn.Embedding(2, embed_dim)
        self.intervention_flag = nn.Embedding(2, embed_dim) 

    def forward(self, m_data):
        u, s, f = m_data.shape
        device = m_data.device
        flat_data = m_data.reshape(-1, 1)
        
        feat_idx = torch.arange(f, device=device).repeat(u * s)
        univ_idx = torch.cat([torch.zeros(s*f), torch.ones((u-1)*s*f)]).long().to(device)
        
        # This tells the model: "In Universe U, Variable U-1 is the CAUSE"
        flags = torch.zeros(u, s, f, device=device)
        for i in range(1, u):
            flags[i, :, i-1] = 1 
        flags = flags.reshape(-1).long()

        return (self.value_encoder(flat_data) + 
                self.feature_embed(feat_idx) + 
                self.universe_embed(univ_idx) + 
                self.intervention_flag(flags)).view(u, s * f, -1)

In [61]:
# --- 4. CROSS-UNIVERSE BLOCK ---
class CrossUniverseBlock(nn.Module):
    def __init__(self, embed_dim, n_heads=4):
        super().__init__()
        self.intra_attention = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.cross_attention = nn.MultiheadAttention(embed_dim, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 4),
            nn.ReLU(),
            nn.Linear(embed_dim * 4, embed_dim)
        )

    def forward(self, u_obs, u_int):
        # Intra-attention (Self-Attention within the Interventional Universe)
        attn_intra, _ = self.intra_attention(u_int, u_int, u_int)
        u_int = self.norm1(u_int + attn_intra)
        # Cross-attention (Interventional Universe queries Observational Universe)
        attn_cross, _ = self.cross_attention(u_int, u_obs, u_obs)
        u_int = self.norm2(u_int + attn_cross)
        return u_int + self.ffn(u_int)

In [62]:
# --- REFINED MULTIVERSE TRANSFORMER --- v2
class MultiverseTransformer(nn.Module):
    def __init__(self, n_features, embed_dim=128, n_heads=8):
        super().__init__()
        self.embedding = ParallelUniverseEmbedding(n_features, embed_dim)
        
        # Encoder for the 'Real World'
        self.obs_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=embed_dim, nhead=n_heads, batch_first=True),
            num_layers=3
        )
        
        # Parallel Universe Decoder Blocks
        self.int_decoder = nn.ModuleList([
            CrossUniverseBlock(embed_dim, n_heads) for _ in range(4) # Deeper for better logic
        ])
        
        self.output_head = nn.Linear(embed_dim, 1)

    def forward(self, m_data):
        u, s, f = m_data.shape
        embedded = self.embedding(m_data)
        
        u_obs = self.obs_encoder(embedded[0].unsqueeze(0))
        u_int = embedded[1:]
        
        # Cross-Universe Reasoning
        obs_context = u_obs.repeat(u-1, 1, 1)
        for layer in self.int_decoder:
            u_int = layer(obs_context, u_int)
            
        return self.output_head(u_int)

# Initialize with the new Logic
model = MultiverseTransformer(n_features=5)

In [63]:
# --- RUN TEST ---
n_feats, n_samps = 5, 10
m_data, adj = generate_full_multiverse(n_samps, n_feats)
model = MultiverseTransformer(n_features=n_feats)

with torch.no_grad():
    predictions = model(m_data)

print(f"Data shape: {m_data.shape}")        # [6, 10, 5]
print(f"Prediction shape: {predictions.shape}") # [5, 50, 1]
print("\nSuccess: The Multiverse Architecture is operational.")

Data shape: torch.Size([6, 10, 5])
Prediction shape: torch.Size([5, 50, 1])

Success: The Multiverse Architecture is operational.


In [64]:


# 2. Refined Training (Higher Samples for clearer signal)
n_features = 5
n_samples = 30 # Increased from 10
model = MultiverseTransformer(n_features=n_features, embed_dim=128)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

print("Starting High-Precision Training...")
for epoch in range(1500):
    # Randomize do_val during training so the model learns 'scaling'
    rand_do = np.random.uniform(2.0, 15.0)
    m_data, _ = generate_full_multiverse(n_samples, n_features, do_val=rand_do)
    
    target = m_data[1:].reshape(n_features, n_samples * n_features, 1)
    
    optimizer.zero_grad()
    preds = model(m_data)
    loss = nn.MSELoss()(preds, target)
    loss.backward()
    optimizer.step()
    
    if epoch % 250 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.6f}")

print("Training Finished. Run your Zero-Shot test now!")

Starting High-Precision Training...
Epoch 0 | Loss: 22.207132
Epoch 250 | Loss: 0.082705
Epoch 250 | Loss: 0.082705
Epoch 500 | Loss: 0.054573
Epoch 500 | Loss: 0.054573
Epoch 750 | Loss: 0.019988
Epoch 750 | Loss: 0.019988
Epoch 1000 | Loss: 1.424389
Epoch 1000 | Loss: 1.424389
Epoch 1250 | Loss: 0.002125
Epoch 1250 | Loss: 0.002125
Training Finished. Run your Zero-Shot test now!
Training Finished. Run your Zero-Shot test now!


In [65]:
def test_blind_inference_final(model, n_samples=10, n_features=5, do_val=10.0):
    """
    Comprehensive blind evaluation with:
    - Loop through all interventions
    - Fixed shape handling
    - DAG recovery metrics (precision/recall)
    - Causal discovery accuracy
    """
    model.eval()
    
    # 1. Generate Ground Truth
    m_true, adj_test = generate_full_multiverse(n_samples, n_features, do_val=do_val)
    
    # 2. Create the Blind Input (only intervened value visible, rest zeroed)
    m_blind = m_true.clone()
    for u_idx in range(1, n_features + 1):
        target_node = u_idx - 1
        for f_idx in range(n_features):
            if f_idx != target_node:
                m_blind[u_idx, :, f_idx] = 0.0

    # 3. Inference
    with torch.no_grad():
        predictions = model(m_blind)  # Shape: [n_features, n_samples * n_features, 1]

    # 4. Extract True DAG Edges
    true_edges = set()
    edges = torch.where(adj_test > 0)
    for j in range(len(edges[0])):
        true_edges.add((edges[0][j].item(), edges[1][j].item()))
    
    print(f"\n{'='*70}")
    print(f"BLIND CAUSAL INFERENCE EVALUATION (n_samples={n_samples})")
    print(f"{'='*70}")
    print(f"True DAG Edges: {sorted(list(true_edges))}")
    
    # 5. Analyze each intervention
    all_pred_edges = {}  # Store discovered edges for each intervention
    
    for u_idx in range(n_features):
        target_node = u_idx
        
        # Reshape predictions correctly: [n_samples, n_features]
        pred_world = predictions[u_idx, :n_samples * n_features, 0].reshape(n_samples, n_features)
        obs_world = m_true[0]
        true_world = m_true[u_idx + 1]
        
        print(f"\n--- Intervention on X{target_node} ---")
        
        # Compute deltas
        true_deltas = (true_world - obs_world).abs().mean(dim=0)
        pred_deltas = (pred_world - obs_world).abs().mean(dim=0)
        
        # Detect causal effects: edges where true delta is significant
        pred_edges_for_this_interv = set()
        detected_threshold = true_deltas.max() * 0.1  # 10% of max delta
        
        print("\nVariable | True Δ | Pred Δ | True Effect?")
        print("-" * 50)
        
        for i in range(n_features):
            true_delta = true_deltas[i].item()
            pred_delta = pred_deltas[i].item()
            
            # Mark as affected if delta exceeds threshold
            has_effect = true_delta > detected_threshold
            
            if has_effect and i != target_node:
                pred_edges_for_this_interv.add((target_node, i))
            
            status = "YES" if has_effect else "NO"
            print(f"X{i}       | {true_delta:6.4f} | {pred_delta:6.4f} | {status}")
        
        all_pred_edges[target_node] = pred_edges_for_this_interv
    
    # 6. Compute DAG Recovery Metrics
    predicted_edges = set()
    for edges_set in all_pred_edges.values():
        predicted_edges.update(edges_set)
    
    # Add direct intervention targets
    for u_idx in range(n_features):
        # The intervention itself should have a direct effect
        predicted_edges.add((u_idx, u_idx))
    
    true_positives = true_edges & predicted_edges
    false_positives = predicted_edges - true_edges
    false_negatives = true_edges - predicted_edges
    
    precision = len(true_positives) / len(predicted_edges) if len(predicted_edges) > 0 else 0
    recall = len(true_positives) / len(true_edges) if len(true_edges) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\n{'='*70}")
    print(f"CAUSAL DISCOVERY METRICS")
    print(f"{'='*70}")
    print(f"Predicted Edges: {sorted(list(predicted_edges))}")
    print(f"True Positives (TP): {len(true_positives)} {sorted(list(true_positives))}")
    print(f"False Positives (FP): {len(false_positives)} {sorted(list(false_positives))}")
    print(f"False Negatives (FN): {len(false_negatives)} {sorted(list(false_negatives))}")
    print(f"\nPrecision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1 Score:  {f1:.4f}")
    print(f"{'='*70}")

# Run evaluation
test_blind_inference_final(model, n_samples=30, n_features=5, do_val=10.0)


BLIND CAUSAL INFERENCE EVALUATION (n_samples=30)
True DAG Edges: [(0, 1), (1, 2), (1, 3), (1, 4)]

--- Intervention on X0 ---

Variable | True Δ | Pred Δ | True Effect?
--------------------------------------------------
X0       | 9.5422 | 9.4191 | YES
X1       | 0.9991 | 0.6774 | YES
X2       | 0.0366 | 0.7045 | NO
X3       | 0.0458 | 1.0395 | NO
X4       | 0.3967 | 0.7777 | NO

--- Intervention on X1 ---

Variable | True Δ | Pred Δ | True Effect?
--------------------------------------------------
X0       | 0.0000 | 0.6704 | NO
X1       | 9.8634 | 9.8097 | YES
X2       | 0.9543 | 0.7073 | NO
X3       | 0.2168 | 1.0432 | NO
X4       | 4.3711 | 0.7730 | YES

--- Intervention on X2 ---

Variable | True Δ | Pred Δ | True Effect?
--------------------------------------------------
X0       | 0.0000 | 0.6620 | NO
X1       | 0.0000 | 0.6762 | NO
X2       | 9.9780 | 10.0458 | YES
X3       | 0.0000 | 1.0417 | NO
X4       | 0.0000 | 0.7736 | NO

--- Intervention on X3 ---

Variable | True Δ | 